# 03 — Baselines (B0 / B1)

Antisymmetric ranking logistic (B0) and surface Elo (B1) with K-by-level updates.
Structural symmetry: `p(A,B) + p(B,A) ≈ 1`.

In [13]:
from tml.features.elo import EloState, expected_score, rating_diff, update_tournament
from tml.models.elo_prob import elo_win_prob
from tml.models.ranking_logit import B0Model
from tml.models.symmetry import logistic
from tml.models.supervised import fit_b0, predict_proba
import pandas as pd
from pathlib import Path

In [14]:
from tml.models.supervised import fit_b0, predict_proba
features = pd.read_parquet(Path("../data/processed/smoke_features.parquet"))
print(features.shape, features["y_complete_win"].value_counts())
train = features.dropna(subset=["y_complete_win"])
b0 = fit_b0(train)
p = predict_proba(b0, train.iloc[[0]])
print(p)

(1053, 22) y_complete_win
1    558
0    495
Name: count, dtype: int64
[0.48876511]


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [ ]:
from tml.features.elo import EloState
from tml.models.elo_prob import elo_win_prob
from tml.models.supervised import predict_proba

state = EloState(k_atp=32.0, k_challenger=20.0)

row = train.iloc[[0]]  # DataFrame with one row, not a Series
modeling = pd.read_parquet(Path("../data/processed/modeling.parquet"))

p_b0 = predict_proba(b0, row)[0]
print("p_b0", p_b0)

r = train.iloc[0]
m = modeling.loc[modeling["match_id"] == r["match_id"]].iloc[0]

p_b1 = elo_win_prob(
    state,
    str(m["player_a_id"]),
    str(m["player_b_id"]),
    str(m["surface"]),
    int(m["best_of"]),
)
p_b1_swap = elo_win_prob(
    state,
    str(m["player_b_id"]),
    str(m["player_a_id"]),
    str(m["surface"]),
    int(m["best_of"]),
)
print("p_b1", p_b1, "sum", p_b1 + p_b1_swap)
assert abs(p_b1 + p_b1_swap - 1.0) < 1e-9


In [17]:
# After scoring a tournament batch, update Elo once (all matches in that tourney).
tournament_matches = modeling.loc[
    modeling["tourney_id"] == m["tourney_id"]
].copy()
print("tournament", m["tourney_id"], "matches", len(tournament_matches))

before = state.get(str(m["player_a_id"]), str(m["surface"]))
update_tournament(state, tournament_matches)
after = state.get(str(m["player_a_id"]), str(m["surface"]))
print("player_a surface Elo before/after:", before, after)


tournament 2018-339 matches 26
player_a surface Elo before/after: 1500.0 1484.0
